# Molecular Design VAE — Basic Mode

**Run cells in order. Total time: ~8 minutes.**

1. Cell 1 — Clone repo + install (~2 min)
2. Cell 2 — Train the model (~5-8 min)
3. Cell 3 — Start server + get live URL
4. Cell 4 — Download trained model (optional)

## Cell 1 — Clone and install

In [ ]:
import os, subprocess

# Clone the repository
subprocess.run(['git', 'clone', 'https://github.com/Kaur-Simarpreet/molecular-design-vae.git'], check=True)

# Change into the repo directory — use os.chdir, not %cd
# os.chdir works for ALL subsequent Python and shell commands
os.chdir('molecular-design-vae')
print('Working directory:', os.getcwd())
print('Files:', [f for f in os.listdir('.') if not f.startswith('.')])

# Install dependencies
subprocess.run(['pip', 'install', '-q', 'torch', 'selfies',
                'flask', 'flask-cors', 'scipy', 'requests'], check=True)
subprocess.run(['pip', 'install', '-q', 'rdkit'], check=True)
print('\nAll dependencies installed successfully')


## Cell 2 — Train the model (~5-8 min)

Trains the basic VAE on 295 curated drug-like molecules.
Saves model to `saved_model/`.

In [ ]:
import os, subprocess

# Confirm we are in the right place
if not os.path.exists('train_vae.py'):
    os.chdir('molecular-design-vae')
print('Directory:', os.getcwd())

# Run training — output prints live
result = subprocess.run(['python', 'train_vae.py'])
if result.returncode != 0:
    raise RuntimeError('Training failed — check output above')

# Confirm model files were saved
required = ['vae.pt', 'tokenizer.pkl', 'latents.pt', 'config.json']
for f in required:
    path = f'saved_model/{f}'
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f'  {path}: {"OK" if exists else "MISSING"} ({size:,} bytes)')


## Cell 3 — Start server + get live URL

No signup. No account. No password.

A URL like `https://abc-xyz.trycloudflare.com` will appear below.
Open it in any browser — all 8 tabs work.

In [ ]:
import os, subprocess, threading, time, re, urllib.request

# Confirm directory
if not os.path.exists('serve.py'):
    os.chdir('molecular-design-vae')
REPO_DIR = os.getcwd()
print('Running from:', REPO_DIR)

# Confirm model files exist
required = ['vae.pt', 'tokenizer.pkl', 'latents.pt', 'config.json']
missing = [f for f in required if not os.path.exists(f'saved_model/{f}')]
if missing:
    raise FileNotFoundError(
        f'Missing model files: {missing}\n'
        'Re-run Cell 2 (training) before this cell.'
    )
print('Model files: OK')

# Start serve.py in background
# cwd=REPO_DIR ensures serve.py finds saved_model/ and index.html
server_proc = subprocess.Popen(
    ['python', 'serve.py', '--host', '0.0.0.0', '--port', '5000'],
    cwd=REPO_DIR
    # No capture_output — errors print to notebook output
)
print('Server process started (PID:', server_proc.pid, ')')
print('Waiting 20 seconds for server to load model...')
time.sleep(20)

# Verify server is actually responding
print('Checking server health...')
for attempt in range(5):
    try:
        r = urllib.request.urlopen('http://localhost:5000/health', timeout=5)
        data = r.read().decode()
        print(f'Server is UP: {data[:120]}')
        break
    except Exception as e:
        print(f'  Attempt {attempt+1}/5 failed: {e}')
        if attempt < 4:
            time.sleep(5)
        else:
            raise RuntimeError(
                'Server failed to start.\n'
                'Check the output above for error messages from serve.py.\n'
                'Most likely cause: training (Cell 2) did not complete.'
            )

# Download cloudflared (no account needed)
print('\nDownloading cloudflared...')
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/tmp/cloudflared'
], check=True)
subprocess.run(['chmod', '+x', '/tmp/cloudflared'])
print('cloudflared ready')

# Start tunnel
tunnel_proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
)

# Read tunnel output until URL appears
print('Starting tunnel — waiting for URL...')
url_found = False
for line in tunnel_proc.stderr:
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m:
            print('\n' + '='*50)
            print('OPEN THIS URL IN YOUR BROWSER:')
            print(m.group(0))
            print('='*50)
            print('All 8 tabs work. No password. No login.')
            print('URL changes each session — copy it now.')
            url_found = True
            break
    # Print other tunnel messages so user can debug if needed
    if 'error' in line.lower() or 'failed' in line.lower():
        print('Tunnel:', line.strip())

if not url_found:
    print('URL not found in tunnel output. Re-run this cell.')


## Cell 4 — Download trained model (optional)

Downloads `saved_model.zip` to your computer.
Use this to run the model locally without retraining.

In [ ]:
import os, shutil
from google.colab import files

if not os.path.exists('saved_model'):
    os.chdir('molecular-design-vae')

shutil.make_archive('/tmp/saved_model', 'zip', 'saved_model')
print('Archive created:', os.path.getsize('/tmp/saved_model.zip'), 'bytes')
files.download('/tmp/saved_model.zip')
print('Download started')
